# 💀 Day 04a: Deadlocks — Detection, Prevention & Avoidance

---

## 🎯 What You'll Master Today
- Coffman's 4 necessary conditions for deadlock
- Resource Allocation Graph (RAG)
- Prevention (break a condition) vs Avoidance (Banker's algorithm)
- Detection (wait-for graph) & recovery

---

```
╔══════════════════════════════════════════════════════════╗
║         COFFMAN'S 4 CONDITIONS (ALL must hold)          ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  1. Mutual Exclusion — resource held exclusively         ║
║  2. Hold and Wait — hold one, wait for another          ║
║  3. No Preemption — can't forcibly take resources       ║
║  4. Circular Wait — P1→P2→P3→P1 cycle                  ║
║                                                          ║
║  Break ANY one → deadlock impossible                     ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Banker's Algorithm — Deadlock Avoidance
# ============================================

def bankers_algorithm(available, max_need, allocation):
    """
    Banker's Algorithm — finds safe sequence.
    available: list of available resources per type
    max_need: matrix[process][resource] — max each process needs
    allocation: matrix[process][resource] — currently allocated
    """
    n = len(max_need)       # number of processes
    m = len(available)      # number of resource types
    
    # Calculate NEED matrix: need[i][j] = max[i][j] - allocation[i][j]
    need = [[max_need[i][j] - allocation[i][j] for j in range(m)] for i in range(n)]
    
    print("Need Matrix:")
    for i in range(n):
        print(f"  P{i}: {need[i]}")
    print(f"  Available: {available}\n")
    
    # Safety algorithm
    work = available[:]     # work = available copy
    finish = [False] * n
    safe_sequence = []
    
    for _ in range(n):
        found = False
        for i in range(n):
            if not finish[i]:
                # Check if need[i] <= work (all resource types)
                if all(need[i][j] <= work[j] for j in range(m)):
                    # Process i can finish
                    print(f"  P{i} can finish. Work {work} + Alloc {allocation[i]} ", end="")
                    work = [work[j] + allocation[i][j] for j in range(m)]
                    print(f"= {work}")
                    finish[i] = True
                    safe_sequence.append(f"P{i}")
                    found = True
                    break
        
        if not found:
            break
    
    if all(finish):
        print(f"\n  ✅ SAFE! Sequence: {' → '.join(safe_sequence)}")
        return True, safe_sequence
    else:
        print(f"\n  ❌ UNSAFE! Deadlock possible.")
        return False, []


# Example: 5 processes, 3 resource types (A, B, C)
available  = [3, 3, 2]
max_need   = [[7,5,3], [3,2,2], [9,0,2], [2,2,2], [4,3,3]]
allocation = [[0,1,0], [2,0,0], [3,0,2], [2,1,1], [0,0,2]]

bankers_algorithm(available, max_need, allocation)

In [ ]:
# ============================================
# 2. Deadlock Detection — Wait-For Graph
# ============================================

def detect_deadlock(wait_for: dict[str, list[str]]) -> list[str]:
    """
    Detect cycle in wait-for graph using DFS.
    wait_for: {process: [processes it waits for]}
    Returns cycle if found, else empty list.
    """
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {p: WHITE for p in wait_for}
    parent = {}
    cycle = []
    
    def dfs(node):
        color[node] = GRAY
        for neighbor in wait_for.get(node, []):
            if color.get(neighbor) == GRAY:  # back edge = cycle!
                # Reconstruct cycle
                cycle.append(neighbor)
                curr = node
                while curr != neighbor:
                    cycle.append(curr)
                    curr = parent.get(curr, neighbor)
                cycle.append(neighbor)
                return True
            if color.get(neighbor, WHITE) == WHITE:
                parent[neighbor] = node
                if dfs(neighbor):
                    return True
        color[node] = BLACK
        return False
    
    for p in wait_for:
        if color[p] == WHITE:
            if dfs(p):
                return cycle
    return []


# Test: P1 waits for P2, P2 waits for P3, P3 waits for P1 → CYCLE!
wait_for = {
    "P1": ["P2"],
    "P2": ["P3"],
    "P3": ["P1"],  # circular wait!
    "P4": ["P2"],  # P4 waits for P2 but is not in cycle
}

cycle = detect_deadlock(wait_for)
if cycle:
    print(f"  💀 DEADLOCK detected! Cycle: {' → '.join(cycle)}")
else:
    print("  ✅ No deadlock")

# No deadlock case
wait_for_safe = {
    "P1": ["P2"],
    "P2": ["P3"],
    "P3": [],
}
cycle = detect_deadlock(wait_for_safe)
print(f"  Safe graph: {'Deadlock' if cycle else 'No deadlock ✅'}")

```
╔══════════════════════════════════════════════════════════════╗
║              DEADLOCK HANDLING STRATEGIES                     ║
╠════════════════════╦═════════════════════════════════════════╣
║  Prevention        ║  Break one of 4 conditions statically  ║
║                    ║  • No hold-and-wait: request all at once║
║                    ║  • No circular wait: resource ordering  ║
║  Avoidance         ║  Don't enter unsafe state              ║
║                    ║  • Banker's algorithm                   ║
║  Detection         ║  Let deadlocks happen, then detect     ║
║                    ║  • Wait-for graph, RAG                  ║
║  Recovery          ║  Kill process or preempt resources      ║
║  Ignore (Ostrich)  ║  Hope it doesn't happen (most OSes!)  ║
╚════════════════════╩═════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Name 4 deadlock conditions | Mutual exclusion, hold & wait, no preemption, circular wait — ALL must hold |
| 2 | Prevention vs avoidance? | Prevention: design rules to break a condition. Avoidance: runtime check (Banker's) |
| 3 | Banker's algorithm complexity? | O(n²m) where n=processes, m=resource types. Check safety for each request |
| 4 | What does Linux do? | Ostrich approach — ignores deadlocks. Relies on programmers and timeouts |
| 5 | How to break circular wait? | Impose total ordering on resources. Always acquire in order (R1 before R2) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • 4 conditions: mutex + hold-wait + no-preempt + cycle │
## │  • Banker's = "is this state safe?" before granting     │
## │  • Wait-for graph: cycle = deadlock                      │
## │  • Most real OSes use ostrich approach + timeouts         │
## │  • Resource ordering is the most practical prevention    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Memory Basics** — contiguous allocation and fragmentation